# Notebook 07 - Evaluate KIRNEWS Classification

This notebook evaluates whether each model can classify Kirundi news articles from KIRNEWS into English category labels.

This is still a proxy task. It is useful because KIRNEWS is labeled, but classification accuracy does not prove broad language quality.

In [ ]:
import sys

from pathlib import Path

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs' / 'project.yaml').exists():
            return candidate
    raise FileNotFoundError(
        'Run this notebook from inside the adaption-kirundi-sft-starter repo.'
    )

ROOT = find_repo_root(Path.cwd())

sys.path.append(str(ROOT / 'src'))

from kirundi_sft_starter.data import load_kirnews_prompts

from kirundi_sft_starter.evals import score_kirnews_predictions
from kirundi_sft_starter.tinker_utils import generate_model_responses

from kirundi_sft_starter.utils import ensure_dir, load_yaml, read_jsonl

project_config = load_yaml(ROOT / 'configs/project.yaml')

eval_config = load_yaml(ROOT / 'configs/evaluation.yaml')

kirnews_labels = eval_config['kirnews_classification']['labels']

In [ ]:
print(kirnews_labels)

## Prepare KIRNEWS prompts

The prompt asks for one English label only. The default label set uses the labels from the dataset card, including `history`, `tourism`, and `fashion`.

In [ ]:
kirnews_prompt_df = load_kirnews_prompts(project_config)

kirnews_prompt_df[['prompt_id', 'gold_label', 'title']].head()

## Generate KIRNEWS predictions

This cell calls Tinker directly from the notebook and writes one prediction file per model under `results/responses/`. Run Notebooks 03 and 04 first so the two SFT checkpoint files exist.

In [ ]:
kirnews_prompts = kirnews_prompt_df.to_dict(orient='records')
kirnews_prediction_files = {
    'base': ROOT / 'results/responses/kirnews_base.jsonl',
    'sft_raw': ROOT / 'results/responses/kirnews_sft_raw.jsonl',
    'sft_adapted': ROOT / 'results/responses/kirnews_sft_adapted.jsonl',
}

for model_key, prediction_path in kirnews_prediction_files.items():
    written_path = generate_model_responses(project_config, model_key, kirnews_prompts, prediction_path)
    print(f'Wrote {written_path}')

In [ ]:
kirnews_prediction_files = {'base': ROOT / 'results/responses/kirnews_base.jsonl', 'sft_raw': ROOT / 'results/responses/kirnews_sft_raw.jsonl', 'sft_adapted': ROOT / 'results/responses/kirnews_sft_adapted.jsonl'}

kirnews_frames = []

for _prediction_model_key, prediction_path in kirnews_prediction_files.items():
    if prediction_path.exists():
        prediction_df = pd.DataFrame(read_jsonl(prediction_path))
        prediction_df['model'] = _prediction_model_key
        prediction_df = prediction_df.rename(columns={'response': 'prediction'})
        kirnews_frames.append(prediction_df)

if not kirnews_frames:
    raise FileNotFoundError('No KIRNEWS prediction files found. Run the prediction generation cell above first.')

kirnews_predictions = pd.concat(kirnews_frames, ignore_index=True)

kirnews_predictions[['model', 'prompt_id', 'gold_label', 'prediction']].head()

In [ ]:
kirnews_rows = []

for _score_model_key, model_group in kirnews_predictions.groupby('model'):
    scores = score_kirnews_predictions(model_group, kirnews_labels)
    kirnews_rows.append({'model': _score_model_key, 'accuracy': round(scores['accuracy'], 4), 'macro_f1': round(scores['macro_f1'], 4), 'num_examples': len(model_group), 'notes': 'proxy automatic classification eval'})

kirnews_summary = pd.DataFrame(kirnews_rows)

kirnews_output_path = ROOT / eval_config['kirnews_classification']['output_path']

ensure_dir(kirnews_output_path)

kirnews_summary.to_csv(kirnews_output_path, index=False)

print('Saved', kirnews_output_path)

print(kirnews_summary.to_string(index=False))

## Suggested qualitative table

After scoring, inspect examples where `sft_adapted` differs from `sft_raw`. Those rows are the most useful for understanding whether Adaption changed the downstream behavior or merely changed surface form.